# NativE (SIGIR 2024) tren MKG-W — kernel chinh, 1000 epoch

Baseline gan APKGC nhat (Bang 2: MRR 0,366 vs 0,374). Ma: github.com/zjukg/NATIVE.
Kernel nay chi chay VAI epoch de do thoi gian moi epoch va thoi gian test,
roi moi dat so epoch cho kernel chinh (gioi han phien Kaggle 12 gio).

Giu nguyen mo hinh va sieu tham so cua scripts/run_mkgw.sh:
dim 250, batch 1024, margin 4, lr 1e-4, lrg 1e-4, neg 128, mu 1e-4, epoch 1000.
NativE test checkpoint CUOI CUNG (khong chon tren test).

In [ ]:
import torch, subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],capture_output=True,text=True).stdout)
print("torch", torch.__version__)
NEED_DOWNGRADE=False
try:
    _=(torch.randn(8,8,device="cuda")@torch.randn(8,8,device="cuda")).sum().item(); print("GPU chay duoc")
except Exception as e:
    print("GPU KHONG chay duoc:", type(e).__name__); NEED_DOWNGRADE=True


In [ ]:
if NEED_DOWNGRADE:
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -2
    print("da ha torch; phan chay dung subprocess nen tien trinh con nap ban moi")


In [ ]:
%cd /kaggle/working
!rm -rf NATIVE && git clone -q https://github.com/zjukg/NATIVE.git
N="/kaggle/working/NATIVE"
%cd $N/mmkgc
!bash make.sh 2>&1 | tail -3
%cd $N
!pip install -q gdown 2>&1 | tail -1
import pathlib, shutil, os, subprocess
os.makedirs("embeddings", exist_ok=True)
def keo(folder, dst):
    r=subprocess.run(["gdown","--folder",folder,"-O",dst],capture_output=True,text=True)
    print((r.stdout+r.stderr)[-1500:])
    moved=[]
    for f in pathlib.Path(dst).rglob("*"):
        if f.suffix==".pth" and f.name.startswith("MKG-W"):
            shutil.copy(str(f), f"embeddings/{f.name}"); moved.append(f.name)
    return moved
NGUON = "NativE Drive 191u4Wh"
got = keo("191u4WhT_7P9Ze8Q9N3U3rdr8IWEKYMry", "emb_native")
if not {"MKG-W-visual.pth","MKG-W-textual.pth"} <= set(got):
    print(">>> Drive cua NativE khong tai du, dung bo nhung MKG-W cua APKGC")
    NGUON = "APKGC Drive 1wGvlzt0"
    got = keo("1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB", "emb_apkgc")
print("NGUON BO NHUNG:", NGUON, "|", sorted(os.listdir("embeddings")))
import torch
for f in sorted(os.listdir("embeddings")):
    t=torch.load("embeddings/"+f, map_location="cpu"); print(f, tuple(t.shape) if hasattr(t,"shape") else type(t))
open("/kaggle/working/NGUON_BO_NHUNG.txt","w").write(NGUON)


In [ ]:
# Script chay: sao chep run_adv_wgan_gp.py cua tac gia, chi them do thoi gian
# va ghi ket qua ra JSON. Mo hinh, loss, sampling, trainer giu nguyen.
RUN = r"""
import json, sys, time, torch
from mmkgc.config import Tester, WCGTrainerGP
from mmkgc.module.model import AdvRelRotatE
from mmkgc.module.loss import SigmoidLoss
from mmkgc.module.strategy import NegativeSamplingGP
from mmkgc.data import TrainDataLoader, TestDataLoader
from mmkgc.adv.modules import CombinedGenerator
from args import get_args
args = get_args()
torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
tr = TrainDataLoader(in_path="./benchmarks/"+args.dataset+"/", batch_size=args.batch_size,
     threads=8, sampling_mode="normal", bern_flag=1, filter_flag=1, neg_ent=args.neg_num, neg_rel=0)
te = TestDataLoader("./benchmarks/"+args.dataset+"/", "link")
img = torch.load("./embeddings/"+args.dataset+"-visual.pth")
txt = torch.load("./embeddings/"+args.dataset+"-textual.pth")
kge = AdvRelRotatE(ent_tot=tr.get_ent_tot(), rel_tot=tr.get_rel_tot(), dim=args.dim,
      margin=args.margin, epsilon=2.0, img_emb=img, text_emb=txt)
model = NegativeSamplingGP(model=kge, loss=SigmoidLoss(adv_temperature=args.adv_temp),
        batch_size=tr.get_batch_size(), regul_rate=0.00001)
gen = CombinedGenerator(noise_dim=64, structure_dim=2*args.dim, img_dim=2*args.dim)
trainer = WCGTrainerGP(model=model, data_loader=tr, train_times=args.epoch, alpha=args.learning_rate,
          use_gpu=True, opt_method="Adam", generator=gen, lrg=args.lrg, mu=args.mu)
STEP = 100
CK = f"./checkpoint/ck-s{args.seed}"
trainer.set_save_steps(STEP, CK)      # luu wrapper moi 100 epoch (co che san cua trainer)
t0=time.time(); trainer.run(); t_train=time.time()-t0
# Test tung checkpoint. Wrapper (NegativeSamplingGP) chua kge nen nap wrapper
# la cap nhat trong so kge. Checkpoint epoch cuoi = dung giao thuc cua NativE.
evals=[]; t1=time.time()
for ep in range(STEP-1, args.epoch, STEP):
    model.load_checkpoint(f"{CK}-{ep}.ckpt")
    mrr, mr, h10, h3, h1 = Tester(model=kge, data_loader=te, use_gpu=True).run_link_prediction(type_constrain=False)
    evals.append(dict(epoch=ep+1, mrr=float(mrr), mr=float(mr), hit1=float(h1), hit3=float(h3), hit10=float(h10)))
    print("EVAL", json.dumps(evals[-1]), flush=True)
t_test=time.time()-t1
fin=evals[-1]; best=max(evals,key=lambda e:e["mrr"])
out=dict(model="NativE", dataset=args.dataset, seed=args.seed, epoch=args.epoch,
         final=fin, best_on_test=best, evals=evals, emb_source=open("/kaggle/working/NGUON_BO_NHUNG.txt").read(),
         train_s=round(t_train,1), test_s=round(t_test,1), s_per_epoch=round(t_train/args.epoch,2))
print("KETQUA_JSON", json.dumps(out))
"""
open(f"{N}/run_native_timed.py","w").write(RUN)
print("ok")

In [ ]:
import subprocess, sys, json, re, os, time
os.makedirs("/kaggle/working/results", exist_ok=True)
def run(seed, epoch):
    cmd=[sys.executable,"run_native_timed.py","-dataset=MKG-W","-batch_size=1024","-margin=4",
         f"-epoch={epoch}","-dim=250",f"-save=./checkpoint/MKG-W-s{seed}-e{epoch}","-neg_num=128",
         "-mu=0.0001","-learning_rate=1e-4","-lrg=1e-4",f"-seed={seed}"]
    t=time.time(); r=subprocess.run(cmd,cwd=N,capture_output=True,text=True)
    log=r.stdout+"\n===STDERR===\n"+r.stderr
    open(f"/kaggle/working/results/native_s{seed}_e{epoch}.log","w").write(log)
    m=re.search(r"KETQUA_JSON (\{.*\})",r.stdout)
    if not m:
        print(f"THAT BAI seed {seed} sau {(time.time()-t)/60:.1f} phut, exit={r.returncode}")
        print("\n".join(log.strip().splitlines()[-25:])); return None
    d=json.loads(m.group(1)); json.dump(d,open(f"/kaggle/working/results/native_s{seed}_e{epoch}.json","w"),indent=1)
    print(json.dumps(d,indent=1)); return d
# KERNEL CHINH: 1 seed, 1000 epoch dung scripts/run_mkgw.sh.
# Do that tu kernel thu: 25,8 s/epoch -> 7,2 gio + 10 lan test x 2,2 phut.
SEED = 42
d = run(SEED, 1000)
if d:
    print(f"\nNativE seed {SEED}: ban cuoi MRR {d['final']['mrr']:.4f} | chon tren test MRR {d['best_on_test']['mrr']:.4f} (epoch {d['best_on_test']['epoch']})")
